# 02 — sympy + numpy (CPU, 2 min)

Dos celdas: (1) identidad trigonométrica con sympy, (2) barrido seeded-500 con numpy + hash.
Copiá las dos líneas JSON al chat de Grafito.

In [ ]:
import json
from sympy import symbols, sympify, simplify
x = symbols("x")
verdict = simplify(sympify("1") - sympify("cos(x)**2+sin(x)**2")) == 0
print(json.dumps({"verdict": bool(verdict), "check": "identity"}))


In [ ]:
import json, struct
import numpy as np
FAMILY = "seeded"
N = 500
SEEDS = [1, 2, 3, 4]
SCALE = 5.0
TOL = 1e-9
MASK = (1 << 64) - 1
SM_GAMMA = 0x9E3779B97F4A7C15
SM_M1 = 0xBF58476D1CE4E5B9
SM_M2 = 0x94D049BB133111EB
def splitmix(state):
    state = (state + SM_GAMMA) & MASK
    z = state
    z = ((z ^ (z >> 30)) * SM_M1) & MASK
    z = ((z ^ (z >> 27)) * SM_M2) & MASK
    return state, z ^ (z >> 31)
def gen_seeded(seed, n, scale):
    st = seed
    pts = np.empty((n, 2))
    for i in range(n):
        st, a = splitmix(st)
        st, b = splitmix(st)
        pts[i, 0] = a / 18446744073709551615 * 2.0 * scale - scale
        pts[i, 1] = b / 18446744073709551615 * 2.0 * scale - scale
    return pts
def unit_count(pts, tol):
    n = len(pts)
    lo, hi = max(0.0, 1.0 - tol), 1.0 + tol
    lo2, hi2 = lo * lo, hi * hi
    total = 0
    CH = 4096
    for s in range(0, n, CH):
        d = pts[s:s + CH, None, :] - pts[None, :, :]
        d2 = __import__("numpy").einsum("ijk,ijk->ij", d, d)
        total += int((__import__("numpy").sum((d2 >= lo2) & (d2 <= hi2))))
    return total // 2
def fnv(data: bytes):
    h = 0xCBF29CE484222325
    for b in data:
        h ^= b
        h = (h * 0x100000001B3) & 0xFFFFFFFFFFFFFFFF
    return h
runs = []
for seed in SEEDS:
    pts = gen_seeded(seed, N, SCALE)
    unit = unit_count(pts, TOL)
    blob = struct.pack("<%dd" % (2 * len(pts)), *pts.ravel())
    runs.append({"seed": seed, "n": len(pts), "unit": unit, "points_hash": format(fnv(blob), "016x")})
print(json.dumps({"runs": runs}))
